# ALIP – run every model in Colab

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (~30–40 min total)
3. The last cell downloads `alip_results.zip` with all predictions and the comparison table — send it back or commit it.

Nothing needs to be installed on your own computer. All models use the same 34-doc held-out test split (24 synthetic + 10 real government forms).

## 0. Setup

In [ ]:
!git clone -q https://github.com/SimbaPonde/ALIP-Mineral-License-Classification.git
%cd ALIP-Mineral-License-Classification
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr poppler-utils zstd pciutils lshw > /dev/null
import shutil; assert shutil.which('tesseract') and shutil.which('pdftoppm'), 'apt install failed: tesseract/poppler missing'
!pip -q install pytesseract pdf2image pdfplumber tabulate
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4 and restart')

In [ ]:
import nltk; nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True)
!python build_unified_dataset.py > /dev/null && echo dataset ready

## 1. Tier 1 — TF-IDF baseline (1 min)

In [ ]:
!python baseline_heldout.py
!python baseline_heldout.py --mask-class-names

## 2. Tier 2 — LayoutLMv3 fine-tuning (~15 min)

In [ ]:
import os
!python layoutlmv3_pipeline.py ocr > /dev/null && echo OCR done
assert len(os.listdir('data/ocr_cache')) == 340, 'OCR failed — see error above'
!python layoutlmv3_pipeline.py train --epochs 15 --lr 1e-5 --batch-size 4 --patience 4
assert os.path.exists('models/layoutlmv3_best/train_history.json'), 'training failed — see error above'
!python layoutlmv3_pipeline.py predict --split test
assert os.path.exists('results/predictions/layoutlmv3_fine_tuned__test.json'), 'prediction failed — see error above'

In [ ]:
# free GPU memory for Llama
import gc; gc.collect(); torch.cuda.empty_cache()

## 3. Tier 3 — Llama 3.1 8B via Ollama (~10 min incl. 5 GB download)

In [ ]:
# Install Ollama (output shown so any failure is visible)
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh -o /tmp/ollama_install.sh && sh /tmp/ollama_install.sh 2>&1 | tail -5

import os, shutil, subprocess, time, urllib.request
OLLAMA = shutil.which('ollama') or next((p for p in ['/usr/local/bin/ollama', '/usr/bin/ollama'] if os.path.exists(p)), None)
assert OLLAMA, 'Ollama did not install — copy the lines printed above and send them over'
print('ollama:', OLLAMA)

# start the server and wait until it answers
subprocess.Popen([OLLAMA, 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open('/tmp/ollama.log').read()[-2000:])
print('server up')
!{OLLAMA} pull llama3.1:8b 2>&1 | tail -1
!{OLLAMA} list

In [ ]:
!python llm_evaluation.py --mode zero-shot | tail -4
!python llm_evaluation.py --mode few-shot | tail -4
import glob; assert len(glob.glob('results/predictions/llama*__test.json')) == 2, 'Llama run failed — see error above'

## 4. Compare all models

In [ ]:
!python compare_models.py
from IPython.display import Image, Markdown, display
display(Image('results/comparison/macro_f1_test.png'))
display(Markdown(open('results/comparison/comparison_test.md').read()))

In [ ]:
!zip -qr alip_results.zip results/predictions results/comparison results/llm_cache models/layoutlmv3_best/train_history.json
from google.colab import files; files.download('alip_results.zip')